# FastAPI Introduction with Data Science Applications
This notebook provides an introduction to FastAPI, a modern web framework for building APIs with Python 3.7+. 
We will explore its integration with data science tools like Pandas and cover aspects like handling requests and validation.

## Setup
Install necessary packages: FastAPI for API development, Uvicorn as an ASGI server, and Pandas for data manipulation.

#### Install packages by syncing the project

Open the Terminal / bash and run 
`uv sync`

## Basic FastAPI Application
Creating a simple FastAPI application with a single route that returns a JSON response. Either run following code or copy it into a python file called `main.py`

In [ ]:
%%file examples/1_plain_fast.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
async def read_root():
    return {"Hello": "World"}


## Running the FastAPI Application
To run the FastAPI application, use the following command in your terminal:

```bash
uvicorn filename:app --reload
```

Replace `filename` with the name of your Python file containing the FastAPI app. The `--reload` flag is useful during development as it will automatically reload the server when you make changes to your code.


### ...since we are using uv

If you don't activate the virtual environment before running the cli command, you need prefix execution with `uv run`

```bash
uv run uvicorn filename:app --reload
```

For example, if your file is named `main.py`, the command would be:

```bash
uv run uvicorn main:app --reload
```

Open your browser and navigate to `http://127.0.0.1:8000` (or have a look the ports and running server published by the codespace) to see the running application. You can also access the interactive API documentation at `http://127.0.0.1:8000/docs`.


### Running inside codespace

In case you're running the FastAPI app inside Codespace, you need to make the app accessible on all network interfaces, which is necessary for Codespace environments.

```bash
uvicorn filename:app --host 0.0.0.0 --port 8000 --reload
```

# Running the FastAPI application with a main function

If you want to run the FastAPI application from a Python script, you can define a `main` function that starts the Uvicorn server. This is useful when you want to run the application programmatically.

You can then run the script using `python filename.py`.


```python
...
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
```

In [ ]:
%%file examples/2_plain_fast_main.py
from fastapi import FastAPI
import uvicorn

app = FastAPI()

@app.get("/")
async def read_root():
    return {"Hello": "World"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

## Swagger API Documentation

FastAPI automatically generates an interactive API documentation (Swagger UI) that lets you explore the available endpoints and interact with them directly.

You can access the Swagger UI at the `/docs` endpoint of your FastAPI application. For example, if your application is running locally, you can access it at `http://localhost:8000/docs`.

With Swagger you can see the available endpoints, make requests, and view the responses directly in your browser.

<img src="images/swagger_img.png">

There is also a JSON version of the API documentation available at the `/openapi.json` endpoint.

## Paths and Parameters

You can define different paths and parameters in FastAPI to create more complex APIs. Here's an example of a path with a parameter:

```python

@app.get("/hello/{name}")
def root(name: str):
    return {f"hello {name}"}
```

In this example, the path `/hello/{name}` contains a parameter `name`. When you make a request to `/hello/world`, the value of `name` will be `"world"`.

### Automatic type validation
FastAPI automatically validates the type of the parameter based on the type annotation. If the type doesn't match, it will return a 422 error with a detailed error message.


In [ ]:
%%file examples/3_fast_main.py

from fastapi import FastAPI
import pandas as pd
import uvicorn

app = FastAPI()

@app.get("/")
async def read_root():
    return {"Hello": "World"}


@app.get("/hello/{name}")
def hello_path(name: str):
    return {"hello" : f"{name}"}


@app.get("/square/{num}")
def get_square(num: int):
    return {"result" : num * num}


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

### Query Parameters

You can also define query parameters in FastAPI by adding parameters to the endpoint function with default values.

```python
@app.get("/items/")
def read_item(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}
```

In [ ]:
%%file examples/4_fast_main.py

from fastapi import FastAPI
import uvicorn

app = FastAPI()

@app.get("/")
async def read_root():
    return {"Hello": "World"}

@app.get("/hello/{name}")
def hello_path(name: str):
    return {"hello" : f"{name}"}


@app.get("/square/{num}")
def get_square(num: int):
    return {"result" : num * num}


@app.get("/items/")
def read_item(skip: int = 0, limit: int = 10):
    # random df with 100 entries
    # return based on skip and limit
    df = pd.DataFrame({"entries": range(100)})
    return df.iloc[skip:skip+limit]

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
    

### Use HTTP Status Codes

You can return different HTTP status codes from your FastAPI endpoints to indicate the success or failure of a request.

```python

from fastapi import HTTPException

@app.get("/items/{item_id}")
def read_item(item_id: int):
    if item_id == 0:
        raise HTTPException(status_code=HTTP_404_NOT_FOUND, detail="Item not found")
    return {"item_id": item_id}

```

In this example, if the `item_id` is `0`, the endpoint will return a `404 Not Found` error with the message `"Item not found"`.
You should use Status Codes in your API responses to provide meaningful information about the request outcome.


## Respond with binary files, like images

You can also return binary files from FastAPI endpoints, such as images or other media files.

```python
from fastapi.responses import FileResponse

@app.get("/image/")
def get_image():
    return FileResponse("img.png")

```

### POST Method, Request Parameters and JSON

You can also define POST methods in FastAPI to receive data from the client. You can define request parameters by directly adding them to the endpoint function, or you can use Pydantic models for more complex data structures.

The advantage of using Pydantic models is that FastAPI will automatically validate the request data based on the model schema.

```python

from pydantic import BaseModel

class Item(BaseModel):
    name: str
    description: str = None
    price: float
    tax: float = None

@app.post("/items/")
async def create_item(item: Item):
    df.append(item.dict(), ignore_index=True)
    return {"item": item}
```

In [ ]:
%%file examples/5_fast_main.py
from fastapi import FastAPI, HTTPException
import uvicorn
import pandas as pd
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    description: str = None
    price: float
    tax: float = None

app = FastAPI()
## static list to store items
df = pd.DataFrame(columns=["name", "description", "price", "tax"])

## endpoint to add items to the df
@app.post("/items/")
async def create_item(item: Item):
    df.loc[len(df)] = pd.Series(item.dict())
    return {"item": item}


##also add an endpoint to get all items and search by name
@app.get("/items/")
def get_items(name: str=None):
    if name:
        return df[df["name"] == name]
    return df


## get items with path params, search by name
@app.get("/items/{name}")
def get_item(name: str):
    filtered_df = df[df["name"] == name]
    if not filtered_df.empty:
        # Convert the first matched row to a dictionary
        return filtered_df.iloc[0].to_dict()
    else:
        raise HTTPException(status_code=404, detail="Item not found")


## python main entry point

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

## Ready to serve a ml model

So much for the basics, let's move on to more advanced topics. In our case we will use FastAPI to build a simple machine learning model and serve it as an API. 

Usually when dealing with machine learning models, we are dealing with computation heavy tasks that can block incoming requests. To avoid this, we can run the model in a background process and use FastAPI to handle incoming requests.

```python
from fastapi import FastAPI, BackgroundTasks


## endpoint with background task
@app.post("/predict/")
async def predict(item: Item, background_tasks: BackgroundTasks):
    background_tasks.add_task(run_model, item)
    return response # usually a message that the model is running

def run_model(item: Item):
    # run the model
    return prediction
```

In [ ]:
%%file examples/6_fast_main.py

from typing import Optional

from fastapi import FastAPI, BackgroundTasks
from fastapi.requests import Request
from pydantic import BaseModel
import uvicorn


class ImageRequest(BaseModel):
    prompt: str


app = FastAPI()

# Function to be run as a background task.
# This is just a placeholder function for demonstration.
# In your application, this could be a function that generates an image.
def write_log(message: str):
    # Example of a time-consuming task: Writing a message to a file.
    # Replace this with the logic of your image generation task.
    with open("log.txt", "a") as file:
        file.write(f"{message}\n")


@app.post("/item")
async def root(image_request: ImageRequest, background_tasks: BackgroundTasks):
    prompt = image_request.prompt
    background_tasks.add_task(write_log, prompt)
    return {"message": "Image generation has started."}


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)